# Сводная таблица результатов experiments

Ноутбук собирает метрики из файлов `exp_*/result.csv` и показывает единую таблицу.

In [1]:
from pathlib import Path
import csv
import pandas as pd

exp_dir = Path().resolve()
experiments_dir = exp_dir if exp_dir.name == "experiments" else (exp_dir / "experiments")

columns = [
    "experiment_id",
    "experiment_name",
    "model",
    "accuracy",
    "f1_macro",
    "f1_bad",
    "roc_auc",
    "precision_bad",
    "recall_bad",
    "notes",
    "num_params",
    "train_time_sec",
]

rows = []
for p in sorted(experiments_dir.glob("exp_*/result.csv")):
    try:
        with open(p, "r", encoding="utf-8") as f:
            r = csv.DictReader(f)
            row = next(r, None)
        if row is not None:
            rows.append({k: row.get(k, "") for k in columns})
    except Exception:
        continue

df = pd.DataFrame(rows, columns=columns)
for c in ["accuracy", "f1_macro", "f1_bad", "roc_auc", "precision_bad", "recall_bad", "num_params", "train_time_sec"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df

,experiment_id,experiment_name,model,accuracy,f1_macro,f1_bad,roc_auc,precision_bad,recall_bad,notes,num_params,train_time_sec
0,exp_01_mfcc_svm,SVM на MFCC,SVM (RBF),0.697842,0.668010,0.568493,0.752167,0.528662,0.614815,"n_mfcc=20, GridSearchCV {'clf__C': 1.0, 'clf__...",NaN,27.141249
1,exp_mfcc_02_rf,Random Forest на MFCC,Random Forest,0.738609,0.678752,0.540084,0.734200,0.627451,0.474074,"GridSearchCV {'max_depth': 10, 'min_samples_le...",NaN,1377.412405
2,exp_03_vggish_ml,VGGish + SVM,VGGish (agg) + SVC (RBF),0.721823,0.692384,0.597222,0.789020,0.562092,0.637037,"VGGish mean+std per file (256 dim), GridSearch...",NaN,94.530350
3,exp_04_1d_cnn,1D CNN на waveform,WaveformCNN1d,0.741007,0.688651,0.560976,0.764460,0.621622,0.511111,"raw waveform 10s 16kHz, 4 blocks Conv1d 32-64-...",133554.0,1176.304042
4,exp_05_2d_cnn,2D CNN на mel-спектрограмме,2D CNN (mel),0.707434,0.673181,0.567376,0.753086,0.544218,0.592593,"n_mels=80, n_frames=320, epochs=50, batch=32",440018.0,57.081881
5,exp_06_lstm,BiLSTM по MFCC,BiLSTM (MFCC),0.702638,0.656380,0.530303,0.717008,0.542636,0.518519,"n_mfcc=20, max_frames=320, hidden=128, layers=...",549394.0,112.159739
6,exp_07_bilstm_gru,BiLSTM-GRU на log-mel,BiLSTM-GRU (log-mel),0.748201,0.708436,0.600760,0.749435,0.617188,0.585185,"n_mels=80, max_frames=320, lstm=128, gru=64, e...",277010.0,49.581868
7,exp_08_cwt_cnn,CWT scalogram + CNN,CWT 2D CNN,0.640288,0.547290,0.342105,0.583123,0.419355,0.288889,PyWavelets CWT morl,429778.0,115.956430
8,exp_09_cnn_transformer,CNN+Transformer (AutoDEAP-style),CNN+Transformer (mel),0.470024,0.469426,0.451613,0.533727,0.339552,0.674074,"n_mels=80, n_frames=320, d_model=128, 2 layers...",949586.0,169.328687
9,exp_10_dsscnet,DSSCNet (SE+Res) на mel,DSSCNet,0.745803,0.716042,0.624113,0.797400,0.598639,0.651852,"n_mels=80, n_frames=320, base_c=64, 4 SE-Res b...",304306.0,55.540082


In [2]:
#Лучшие эксперименты
df_top_5_f1_macro = df.nlargest(n=5, columns=['f1_macro'])
df_top_5_f1_bad = df.nlargest(n=5, columns=['f1_bad'])
df_top_5_accuracy = df.nlargest(n=5, columns=['accuracy'])
pd.concat([df_top_5_accuracy, df_top_5_f1_macro, df_top_5_f1_bad]).drop_duplicates()

,experiment_id,experiment_name,model,accuracy,f1_macro,f1_bad,roc_auc,precision_bad,recall_bad,notes,num_params,train_time_sec
28,exp_30_whisper_svm,Whisper encoder SVM,Whisper embedding + SVM,0.829736,0.799472,0.721569,0.890334,0.766667,0.681481,"whisper_small_encoder/grid={'clf__C': 1.0, 'cl...",NaN,3764.442954
12,exp_13_hubert,HuBERT + LSTM,HubertLSTMClassifier,0.798561,0.745080,0.628319,0.823207,0.780220,0.525926,"facebook/hubert-base-ls960, encoder frozen, Bi...",95291794.0,9297.916253
21,exp_23_multilingual_framework,Multilingual XLS-R framework,XLS-R embedding + SVM,0.791367,0.747384,0.641975,0.832650,0.722222,0.577778,"xlsr300m/grid={'clf__C': 3.0, 'clf__gamma': 's...",NaN,NaN
17,exp_18_openl3,OpenL3 + SVM,SVM (RBF),0.779376,0.749974,0.664234,0.817061,0.654676,0.674074,"embedding=OpenL3, agg=mean+max",NaN,249.658258
20,exp_22_swin_mel,Swin on mel,Swin Tiny,0.769784,0.717782,0.596639,0.812582,0.689320,0.525926,"mel resized 224x224, timm",27520908.0,264.873180
32,exp_34_multivowel_voice_disorders,Multicategory voice disorders from mel,MelCNN2d,0.767386,0.721020,0.607287,0.778487,0.669643,0.555556,mel 2D CNN proxy,440018.0,72.082154
9,exp_10_dsscnet,DSSCNet (SE+Res) на mel,DSSCNet,0.745803,0.716042,0.624113,0.797400,0.598639,0.651852,"n_mels=80, n_frames=320, base_c=64, 4 SE-Res b...",304306.0,55.540082


In [3]:
df.to_csv(experiments_dir / "metrics_summary.csv", index=False, encoding="utf-8")